# 10 · Trajectory models: gradient boosting vs LSTM (P4)

Gradient boosting and an LSTM against the baselines, a learning curve (how much data each model needs), the chosen model tested once and its uncertainty band. Code: `trajectory/models.py`, `trajectory/experiment.py`. Reads `outputs/trajectory/trajectory_results.json` (`python -m recommendation.main trajectory` retrains, ~1 minute).

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
r = json.loads((OUTPUTS / "trajectory" / "trajectory_results.json").read_text())
print("Chosen:", r["chosen"], "| best baseline:", r["best_baseline"])
pd.DataFrame({n: {h: c[h]["mae"] for h in ("24m", "36m", "60m", "all")} for n, c in r["cv"].items()}).T.round(0)

## 1. Test grafts (used once)

In [ ]:
display(pd.DataFrame({n: {h: d["errors"][h]["mae"] for h in ("24m", "36m", "60m", "all")}
                      for n, d in r["test"].items()}).T.round(0))
print(r["band_level"], "band coverage on test:",
      {h: f'{d["coverage"]:.0%}' for h, d in r["test"][r["chosen"]]["band_coverage"].items()})

## 2. Learning curve: does more data help, and when does the LSTM pay off?

In [ ]:
curve = pd.DataFrame(r["learning_curve"]).groupby(["train_grafts", "model"])["mae_all"].mean().unstack()
display(curve.round(0))
display(Image(OUTPUTS / "trajectory" / "learning_curve.png"))

## 3. The saved model on three test grafts

In [ ]:
import joblib
from recommendation.data_prep.followup_dataset import load

table, _ = load(P["followup_dataset"], MODELS / "followup_preprocessor.json")
traj = joblib.load(MODELS / "followup_trajectory.joblib")
test = table[table["split"] == "test"].head(3)
pd.concat([test[["graft_id", "procedure", "ECD_12m", "ECD_24m", "ECD_36m", "ECD_60m"]],
           traj["model"].predict(test).round(0).add_prefix("pred_")], axis=1)